## Pre-requisites:
https://www.youtube.com/watch?v=eMlx5fFNoYc

## Introduction

**FlashAttention-3 (FA3)** combined with **Scaled Dot-Product Attention (SDPA)** represents the cutting-edge of GPU-accelerated attention mechanisms, specifically optimized for NVIDIA Hopper architecture (H100/H200 GPUs) to significantly speed up LLM training and inference.

- <a href='../papers/4_FLASH_ATTENTION.pdf'>Flash Attention</a>
- <a href='../papers/5_FLASH_ATTENTION_2.pdf'>Flash Attention2</a>
- <a href='../papers/6_FLASH_ATTENTION_3.pdf'>Flash Attention3</a>

It is a specialized, faster successor to FlashAttention-2 (FA2) that leverages new hardware features on Hopper, such as Tensor Cores and asynchronous memory management, to achieve1.5–2× higher speed than FA2.

### Key Aspects of FA3/SDPA
- **Hopper Optimization**: FA3 is designed specifically for NVIDIA Hopper (sm90a) GPUs, utilizing asynchronous operations to overlap computation and memory movement.

- **Performance Gains**: FA3 offers 1.5–2.0× faster forward passes and 1.5–1.75× faster backward passes compared to FA2.

- **PyTorch Integration**: FA3 is being integrated as a backend for torch.nn.functional.scaled_dot_product_attention (SDPA), which automatically selects the fastest available kernel.

- **Limitations**: Unlike FA2, FA3 currently has limited support for dropout and is not fully backward-compatible with older GPU architectures.

- **Key Features**: It supports Grouped Query Attention (GQA), Multi-Query Attention (MQA), and variable-length sequences (varlen). 

### FA3 vs. FA2 vs. Other Kernels
- FA3: Fastest on H100, optimized for asynchronous execution.
- FA2: Generally used for architectures older than Hopper (e.g., A100).
- cuDNN: A fast alternative to FA3 that supports wider, more arbitrary attention masks (sliding windows) but may be slightly slower than FA3 on Hopper. 

Tiling in machine learning refers to breaking large datasets or complex, high-dimensional tensors into smaller, manageable chunks (tiles) to fit into limited memory, such as GPU cache. It enhances computational performance, increases data locality, reduces cache misses, and enables parallel processing of large-scale images or matrices, crucial for training deep learning models. 

In [55]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

In [56]:
vocab_size: int = 10
n_embd: int = 5
block_size: int = 3

In [57]:
wte = nn.Embedding(vocab_size, n_embd) # token embedding
wpe = nn.Embedding(block_size, n_embd)# position embedding

wte, wpe

(Embedding(10, 5), Embedding(3, 5))

In [58]:
wte.weight, wpe.weight

(Parameter containing:
 tensor([[-0.6866, -0.2764, -0.9392, -0.4992,  1.1591],
         [-1.0834, -0.2236,  0.0432, -0.4767, -0.6872],
         [-1.2752,  0.1928, -0.0066,  0.9814, -0.1553],
         [ 0.9677, -0.0353, -0.4628, -0.0213, -1.3923],
         [-0.0702, -1.1400,  0.3631,  0.2930,  0.1040],
         [-0.4066, -0.3570,  1.6541,  1.0632, -0.4705],
         [-0.9288, -1.4043,  0.3024,  0.3684,  0.4810],
         [ 1.7734,  2.3836, -0.1052,  0.0993,  0.1874],
         [-0.3831, -0.1978,  1.0145, -1.1994, -0.1222],
         [-0.2395, -0.5295,  0.2448,  1.3301, -0.2483]], requires_grad=True),
 Parameter containing:
 tensor([[-0.2455,  0.3586,  1.5658,  1.4269,  0.1528],
         [ 0.4988,  0.2699,  0.5631,  0.5654, -0.3003],
         [-0.4512, -0.4969, -0.9905, -0.3628,  2.8745]], requires_grad=True))

In [59]:
idx = torch.randint(0, vocab_size, (2, 3))
B, T = idx.size()
B, T, idx

(2,
 3,
 tensor([[0, 5, 2],
         [7, 0, 4]]))

In [60]:
pos = torch.arange(0, T, dtype=torch.long) # shape (T)
pos_emb = wpe(pos) # shape (T, n_embd)

pos, pos_emb, pos_emb.shape

(tensor([0, 1, 2]),
 tensor([[-0.2455,  0.3586,  1.5658,  1.4269,  0.1528],
         [ 0.4988,  0.2699,  0.5631,  0.5654, -0.3003],
         [-0.4512, -0.4969, -0.9905, -0.3628,  2.8745]],
        grad_fn=<EmbeddingBackward0>),
 torch.Size([3, 5]))

In [61]:
tok_emb = wte(idx)
tok_emb, tok_emb.shape

(tensor([[[-0.6866, -0.2764, -0.9392, -0.4992,  1.1591],
          [-0.4066, -0.3570,  1.6541,  1.0632, -0.4705],
          [-1.2752,  0.1928, -0.0066,  0.9814, -0.1553]],
 
         [[ 1.7734,  2.3836, -0.1052,  0.0993,  0.1874],
          [-0.6866, -0.2764, -0.9392, -0.4992,  1.1591],
          [-0.0702, -1.1400,  0.3631,  0.2930,  0.1040]]],
        grad_fn=<EmbeddingBackward0>),
 torch.Size([2, 3, 5]))

In [62]:
x = tok_emb + pos_emb
x, x.shape

(tensor([[[-0.9321,  0.0822,  0.6266,  0.9277,  1.3119],
          [ 0.0922, -0.0871,  2.2171,  1.6286, -0.7708],
          [-1.7265, -0.3041, -0.9971,  0.6186,  2.7192]],
 
         [[ 1.5279,  2.7422,  1.4606,  1.5262,  0.3402],
          [-0.1878, -0.0065, -0.3761,  0.0661,  0.8588],
          [-0.5214, -1.6369, -0.6274, -0.0698,  2.9785]]],
        grad_fn=<AddBackward0>),
 torch.Size([2, 3, 5]))

In [63]:
c_attn = nn.Linear(n_embd, 3 * n_embd)
c_attn, c_attn.weight.shape

(Linear(in_features=5, out_features=15, bias=True), torch.Size([15, 5]))

In [64]:
qkv = c_attn(x)
qkv, qkv.shape

(tensor([[[ 0.7515,  0.2884,  0.4539, -0.6052, -0.6670, -1.1740,  0.1415,
            0.0135, -0.4804, -0.1465,  0.1919,  0.5348, -0.6463, -0.5045,
           -0.0305],
          [ 0.3015,  0.0208,  0.0067,  0.4765,  0.1529, -1.6234,  0.2564,
            0.8689, -1.1825, -0.6669,  0.6558,  0.7388,  0.1203, -0.2036,
           -1.8978],
          [ 1.1029,  0.7777,  0.6297, -1.7987, -1.4346, -0.8633, -0.0687,
           -0.9707,  0.2140,  0.2827,  0.1694,  0.3940, -1.1641, -0.7423,
            1.2939]],
 
         [[-0.0740, -1.1575,  1.5748,  1.1928,  1.4652, -0.3817,  1.5497,
            1.0357, -0.2309, -0.8806, -0.9806, -0.5016,  0.5491, -0.5899,
           -0.8573],
          [ 0.0625,  0.0329,  0.4454, -0.4906, -0.4314, -0.3245, -0.2775,
           -0.2356, -0.0982,  0.1473,  0.1555, -0.1205, -0.0571, -0.3755,
            0.0648],
          [ 0.4979,  0.6272,  0.5041, -1.7253, -1.8208, -1.1365, -0.4512,
           -1.3476,  0.1339,  1.2976,  0.0264,  0.0325, -1.1425, -0.6453,
    

In [65]:
q, k, v = qkv.split(n_embd, dim=2)
q, k, v, q.shape, k.shape, v.shape

(tensor([[[ 0.7515,  0.2884,  0.4539, -0.6052, -0.6670],
          [ 0.3015,  0.0208,  0.0067,  0.4765,  0.1529],
          [ 1.1029,  0.7777,  0.6297, -1.7987, -1.4346]],
 
         [[-0.0740, -1.1575,  1.5748,  1.1928,  1.4652],
          [ 0.0625,  0.0329,  0.4454, -0.4906, -0.4314],
          [ 0.4979,  0.6272,  0.5041, -1.7253, -1.8208]]],
        grad_fn=<SplitBackward0>),
 tensor([[[-1.1740,  0.1415,  0.0135, -0.4804, -0.1465],
          [-1.6234,  0.2564,  0.8689, -1.1825, -0.6669],
          [-0.8633, -0.0687, -0.9707,  0.2140,  0.2827]],
 
         [[-0.3817,  1.5497,  1.0357, -0.2309, -0.8806],
          [-0.3245, -0.2775, -0.2356, -0.0982,  0.1473],
          [-1.1365, -0.4512, -1.3476,  0.1339,  1.2976]]],
        grad_fn=<SplitBackward0>),
 tensor([[[ 0.1919,  0.5348, -0.6463, -0.5045, -0.0305],
          [ 0.6558,  0.7388,  0.1203, -0.2036, -1.8978],
          [ 0.1694,  0.3940, -1.1641, -0.7423,  1.2939]],
 
         [[-0.9806, -0.5016,  0.5491, -0.5899, -0.8573],
     

## SDPA Attention - SWAT (Sliding Window Attention Training)


https://github.com/Fzkuji/swat-attention

<img src='./assets/sliding_window_attn.png' width='55%'/>

With a window size of ω ≪ N, the computation cost per token is reduced to O(ω), leading to an overall linear complexity O(N · ω), which is more efficient than vanilla attention.

- In the figure above, the window size is three (ω = 3) and the depth is two (L = 2). We define the tokens that are visible to the current window as active tokens
(the red block in the figure, corresponding active tokens are “a dear little”).

- For invisible tokens, also referred to as Evicted tokens we further categorize them as residual and past tokens. Residual tokens are not visible to the sliding window at
the embedding layer

- However, their information will passed to the neighboring ω − 1 tokens with a transformer layer (this information transition is represented as yellow lines in the figure), thus partially preserved for the prediction. 

- For example, the information of the token ‘a’ (the orange ball at the embedding layer) can be retained in the other token ‘a’ (the red ball at the second transformer layer) in our visualization. Theoretically, the information range of a single token at the lth transformer layer is 1 + (ω − 1) · l and the maximum range is 1 + (ω − 1) · L, 
i.e., 1 + 2 · 2 = 5 in the figure.

In [ ]:
@dataclass
class GPTConfig:
    sequence_len: int = 8
    vocab_size: int = 16
    n_layer: int = 4
    n_head: int = 2 # number of query heads
    n_kv_head: int = 2 # number of key/value heads (GQA)
    n_embd: int = 4
    # Sliding window attention pattern string, tiled across layers. Final layer always L.
    # Characters: L=long (full context), S=short (half context)
    # Examples: "L"=all full context, "SL"=alternating, "SSL"=two short then one long
    window_pattern: str = "SSSL"

In [67]:
def compute_window_sizes(config: GPTConfig):
  '''
  compute per-layer window sizes for sliding window attention

  Returns list of (left, right) tuples for FA3's window_size parameter:
  - left: how many tokens before current position to attend to (-1 = unlimited)
  - right: how many tokens after current position to attend to (0 for causal)
  Pattern string is tiled across layers. Final layer always gets the full context (L)
  Characters: L = Long(full-context), S = Short(half-context)
  '''

  pattern = config.window_pattern.upper()
  assert all(c in "SL" for c in pattern), f"Invalid window_pattern: {pattern}. Use only S and L."

  # map chars to window sizes
  long_window = config.sequence_len
  short_window = long_window // 2
  char_to_window = {
    'L': (long_window, 0),
    'S': (short_window, 0)
  }

  # Tile pattern across layers
  window_sizes = []
  for layer_idx in range(config.n_layer):
    char = pattern[layer_idx % len(pattern)]
    window_sizes.append(char_to_window[char])
  
  # Final layer always gets full context
  window_sizes[-1] = (long_window, 0)
  return window_sizes


In [68]:
compute_window_sizes(config=GPTConfig)

[(4, 0), (4, 0), (4, 0), (8, 0)]

In [69]:
r = torch.arange(5).unsqueeze(1)
c = torch.arange(5).unsqueeze(0)
mask = c <= r
r, c, mask

(tensor([[0],
         [1],
         [2],
         [3],
         [4]]),
 tensor([[0, 1, 2, 3, 4]]),
 tensor([[ True, False, False, False, False],
         [ True,  True, False, False, False],
         [ True,  True,  True, False, False],
         [ True,  True,  True,  True, False],
         [ True,  True,  True,  True,  True]]))

In [70]:
mask_inter =  (r - c) <= 2 # 2 is the window size
mask_inter

tensor([[ True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True],
        [False,  True,  True,  True,  True],
        [False, False,  True,  True,  True]])

In [71]:
mask = mask & mask_inter
mask

tensor([[ True, False, False, False, False],
        [ True,  True, False, False, False],
        [ True,  True,  True, False, False],
        [False,  True,  True,  True, False],
        [False, False,  True,  True,  True]])

In [72]:
def sdpa_attn(q, k, v, window_size, enable_gqa):
  '''
    SDPA attention with sliding window support
    q, k, v are (B, H, T, D) format.

    B (Batch Size): The number of sequences processed in parallel.
    H (Number of Heads): The number of attention heads.
    T (Sequence Length / Time): The length of the input sequence (sometimes denoted as S or N).
    D (Head Dimension): The dimension of each attention head d_head. 
  '''
  Tq = q.size(2) # 8
  Tk = k.size(2) # 8
  window = window_size[0] # 4

  # Full context, same length
  if (window < 0 or window >= Tq) and Tq == Tk:
     print('Full context')
     return F.scaled_dot_product_attention(q, k, v, is_causal=True, enable_gqa=enable_gqa)
  
  # Single token generation
  if Tq == 1:
    print('Single Token Generation')
    if window >= 0 and window < Tk:
      # window is "left" tokens we need to include (window + 1) keys total
      start = max(0, Tk - (window + 1))
      k = k[:, :, start:, :]
      v = v[:, :, start:, :]

    return F.scaled_dot_product_attention(q, k, v, is_causal=False, enable_gqa=enable_gqa)
  
  # Need explicit mask for sliding window/chunk inference
  row_idx = (Tk - Tq) + torch.arange(Tq).unsqueeze(1) # (8, 1)
  col_idx = torch.arange(Tk).unsqueeze(0) # (1, 8)
  mask = col_idx <= row_idx

  # sliding window (left)
  if window >= 0 and window < Tk:
    mask = mask & ((row_idx - col_idx) <= window)
  
  return F.scaled_dot_product_attention(q, k, v, attn_mask=mask, enable_gqa=enable_gqa)

In [73]:
def flash_attn_func(q, k, v, causal=False, window_size=(-1, -1)):
  '''
    Flash attention for training (no KV cache).

    Args:
      q, k, v: tensors of shape (B, T, H, D)
      causal: whether to use causal masking
      window_size: (left, right) sliding window. -1 means unlimited.
    
    Returns:
      output tensor of shape (B, T, H, D)
  '''
  
  # SDPA : transpose (B, T, H, D) -> (B, H, T, D)
  q = q.transpose(1, 2)
  k = k.transpose(1, 2)
  v = v.transpose(1, 2)
  enable_gqa = q.size(1) != k.size(1)
  print(f'{enable_gqa=}')
  y = sdpa_attn(q, k, v, window_size, enable_gqa)
  return y.transpose(1, 2)  # back to (B, T, H, D)

In [74]:
q = torch.randn(2, GPTConfig.sequence_len, GPTConfig.n_head, GPTConfig.n_embd)
k = torch.randn(2, GPTConfig.sequence_len, GPTConfig.n_head, GPTConfig.n_embd)
v = torch.randn(2, GPTConfig.sequence_len, GPTConfig.n_head, GPTConfig.n_embd)
print(q)
print('-' * 77)
print(k)
print('-' * 77)
print(v)

tensor([[[[ 1.5844e+00,  8.1819e-01, -9.5736e-01,  4.7341e-01],
          [ 7.2906e-02, -1.0255e-01,  1.5499e+00, -6.7717e-01]],

         [[ 1.1907e+00, -8.3216e-01,  4.9940e-01, -8.2491e-01],
          [-3.6700e-01, -1.7001e+00,  2.1210e+00,  4.3540e-01]],

         [[ 7.5507e-01,  4.3304e-01,  6.3267e-01, -3.5090e-01],
          [-2.1294e+00,  4.6324e-01, -2.7640e+00,  2.1709e-01]],

         [[ 1.3062e+00,  1.1789e+00,  5.7250e-01, -3.1536e-02],
          [-7.9289e-01, -5.5787e-01,  5.0256e-01, -4.4935e-01]],

         [[ 5.0178e-01, -7.3936e-01,  9.0007e-01,  2.2903e-01],
          [ 2.7389e-02,  1.1745e+00, -6.7850e-01, -6.3019e-01]],

         [[-1.1514e+00, -5.7475e-01,  9.9232e-01, -1.5736e+00],
          [-1.2157e+00,  5.8039e-01,  1.1788e+00,  2.8031e-01]],

         [[-2.8169e+00,  3.0556e-01,  2.2159e+00, -4.5097e-01],
          [-8.9224e-01,  2.1246e+00,  3.4061e-01,  1.1072e+00]],

         [[ 6.9311e-01, -6.0034e-03,  4.3808e-01,  5.7503e-01],
          [ 1.3773e+00, -1

In [75]:
flash_attn_func(q, k, v, causal=True, window_size=(4, 0))

enable_gqa=False


tensor([[[[ 0.3929,  0.7272, -1.3369,  0.1063],
          [ 1.5764, -0.2066, -0.3332, -1.5754]],

         [[ 0.0677, -0.4090, -0.5879,  0.6340],
          [ 1.6543, -0.0196,  0.4627,  0.0192]],

         [[-0.0114, -0.0822,  0.1959,  0.1488],
          [ 1.2611, -0.4329, -0.4094, -0.6079]],

         [[-0.0527,  0.0721,  0.4718, -0.0045],
          [ 1.1541, -0.2259, -0.1467, -0.2174]],

         [[-0.4000,  0.1423, -0.2029,  0.3229],
          [ 0.8651, -0.1879, -0.4132, -0.4683]],

         [[-1.6751,  0.1228,  0.2730,  0.5556],
          [ 0.8134, -0.1472, -0.5406, -0.0809]],

         [[-1.3909,  0.4211, -0.0649,  0.3924],
          [ 0.9420, -0.6631, -0.9092, -0.3908]],

         [[-0.7241, -0.0732, -0.1471,  0.7277],
          [ 0.5545,  0.0887, -0.6722, -0.7590]]],


        [[[ 0.2369,  0.2222, -0.4859, -2.1127],
          [ 1.0770,  0.5317,  1.7628, -1.5134]],

         [[ 0.3930,  0.1674, -0.3300, -1.8493],
          [-0.4225,  0.9114,  0.9430, -0.7558]],

         [[ 0.9585

<img src='./assets/1_uyuyOW1VBqmF5Gtv225XHQ.gif' alt='visualization of kv cache' width='75%'/>

https://huggingface.co/blog/not-lain/kv-caching

<img src='./assets/DbL2RbXFRoMWA5CrOaGB8.png' alt='steps of kv cache' width='75%'/>

In [107]:
x = torch.zeros(2, 10, 3, 3)
print(f"Shape of x: {x.shape}")

y = torch.ones(2, 1, 3, 3)
print(f"Shape of y: {y.shape}")

Shape of x: torch.Size([2, 10, 3, 3])
Shape of y: torch.Size([2, 1, 3, 3])


In [109]:
x[:, 5:6, :, :] = y
x

tensor([[[[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]],

         [[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]],

         [[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]],

         [[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]],

         [[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]],

         [[1., 1., 1.],
          [1., 1., 1.],
          [1., 1., 1.]],

         [[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]],

         [[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]],

         [[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]],

         [[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]]],


        [[[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]],

         [[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]],

         [[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]],

         [[0., 0., 0.],
          [0

In [51]:
def flash_attn_with_kvcache(q, k_cache, v_cache, k=None, v=None, cache_seqlens=None, causal=False, window_size=(-1, -1)):
  ''' 
    Flash attention with KV cache for inference.
    FA3 updates k_cache/v_cache in-place. Our SDPA fallback does the same.

    Args:
      q: Queries, shape (B, T_new, H, D)
      k_cache, v_cache: Pre-allocated cache tensors, shape (B, T_max, H_kv, D)
      k, v: New keys/values to insert, shape (B, T_new, H_kv, D)
      cache_seqlens: Current position in cache, shape (B,) int32
      causal: Whether to use causal masking
      window_size: (left, right) sliding window. -1 means unlimited.
    
    Returns:
      Output tensor of shape (B, T_new, H, D)
  '''
  
  # SDPA fallback: manually manage KV cache
  B, T_new, H, D = q.shape
  pos = cache_seqlens[0].item()  # assume uniform position across batch

  # Insert new k, v into cache (in-place, matching FA3 behavior)
  if k is not None and v is not None:
    k_cache[:, pos: pos+T_new, :, :] = k
    v_cache[:, pos: pos+T_new, :, :] = v

  # Get full cache up to current position + new tokens
  end_pos = pos + T_new
  k_full = k_cache[:, :end_pos, :, :]
  v_full = v_cache[:, :end_pos, :, :]

  # Transpose to SDPA layout: (B, T, H, D) -> (B, H, T, D)
  q_sdpa = q.transpose(1, 2)
  k_sdpa = k_full.transpose(1, 2)
  v_sdpa = v_full.transpose(1, 2)

  enable_gqa = q_sdpa.size(1) != k_sdpa.size(1)
  y_sdpa = sdpa_attn(q_sdpa, k_sdpa, v_sdpa, window_size, enable_gqa)

  return y_sdpa.transpose(1, 2)  # back to (B, T, H, D)